
# Notebook 5 — Distributionally-Robust AVaR and the Optimizer's Curse

**Original contribution of this thesis (Section 6.5).** Two ideas from
the required references are combined: Bassi, Embrechts and Kafetzaki's
theory of quantile-estimation uncertainty (Section 2.5), and Föllmer and
Schied's robust representation of coherent risk measures (Section 2.3).

**Proposition 6.1.** If $(\rho_i)_{i\in I}$ are coherent risk measures and
$\rho(X):=\sup_i \rho_i(X)$ is finite for every $X$, then $\rho$ is itself
coherent.

This licenses a *bootstrap-sup* robustified AVaR: resample the data $B$
times, compute $\mathrm{AVaR}_\lambda$ on each resample, and take the
**maximum** across resamples. It is provably coherent. We also consider
the computationally gentler, but *not* provably coherent, **quantile**
version: a fixed confidence-level quantile (e.g. 80%) across bootstrap
replicates rather than their exact maximum.

We then run the **optimizer's curse** experiment: optimise a portfolio
against a *finite sample*'s AVaR, then compare the naive in-sample
estimate, the two robustified estimates, and the *true* (population)
AVaR of that same portfolio (known exactly here since the data-generating
process is Gaussian).


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from convexrisk.robust import (
    bootstrap_avar_replicates, robust_avar_sup, robust_avar_quantile,
    optimizer_curse_experiment,
)
from convexrisk.risk_measures import discrete_avar, gaussian_avar

plt.rcParams["figure.dpi"] = 110



## 1. Proposition 6.1, verified directly against the four axioms

Rather than rely only on the abstract proof, we verify monotonicity,
cash invariance, positive homogeneity and convexity numerically for
$\rho(X) := \max_i \mathrm{AVaR}_{0.2}^{P_i}(X)$ across three different
fixed probability weightings $P_i$ over the same 50 outcomes.


In [ ]:

rng = np.random.default_rng(2)
n_scen = 50
outcomes = rng.normal(0.0, 1.0, size=n_scen)
weight_sets = [
    np.full(n_scen, 1.0 / n_scen),
    rng.dirichlet(np.ones(n_scen)),
    rng.dirichlet(np.full(n_scen, 0.3)),
]

def rho(x_values):
    return max(discrete_avar(x_values, w, 0.2) for w in weight_sets)

X = outcomes
Y = rng.normal(0.5, 1.2, size=n_scen)
Y_dominating = X + np.abs(rng.normal(0, 0.1, size=n_scen))

print("Monotonicity:      rho(Y_dominating) <= rho(X)  ->",
      rho(Y_dominating) <= rho(X) + 1e-9, f"  ({rho(Y_dominating):.4f} <= {rho(X):.4f})")

m = 2.3
print("Cash invariance:    rho(X+m) == rho(X)-m         ->",
      abs(rho(X + m) - (rho(X) - m)) < 1e-8, f"  ({rho(X+m):.4f} vs {rho(X)-m:.4f})")

t = 3.0
print("Positive homogen.:  rho(tX) == t*rho(X)           ->",
      abs(rho(t * X) - t * rho(X)) < 1e-8, f"  ({rho(t*X):.4f} vs {t*rho(X):.4f})")

lam = 0.4
mixed = lam * X + (1 - lam) * Y
lhs, rhs_ = rho(mixed), lam * rho(X) + (1 - lam) * rho(Y)
print("Convexity:          rho(mix) <= lam*rho(X)+(1-lam)*rho(Y)  ->",
      lhs <= rhs_ + 1e-8, f"  ({lhs:.4f} <= {rhs_:.4f})")



## 2. Why the exact sup can overcorrect: the bootstrap replicate distribution

Before running the full experiment, look at the distribution of
bootstrap AVaR replicates for one fixed, small sample: the maximum is a
single draw from the far tail of this distribution and can be an
unrepresentatively extreme value, while a moderate quantile (e.g. 80%)
sits in a more stable, representative region.


In [ ]:

rng = np.random.default_rng(0)
pnl_sample = rng.normal(0.02, 0.15, size=50)  # a small, realistic sample size
level = 0.1
n_bootstrap = 2000

replicates = bootstrap_avar_replicates(pnl_sample, level, n_bootstrap, rng)
naive_estimate = discrete_avar(pnl_sample, np.full(50, 1/50), level)
sup_value = replicates.max()
q80_value = np.quantile(replicates, 0.8)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(replicates, bins=60, color="C0", alpha=0.75)
ax.axvline(naive_estimate, color="k", linestyle="-", label=f"naive (full sample) = {naive_estimate:.3f}")
ax.axvline(q80_value, color="C1", linestyle="--", label=f"80% quantile = {q80_value:.3f}")
ax.axvline(sup_value, color="C3", linestyle="--", label=f"exact max = {sup_value:.3f}")
ax.set_xlabel(r"bootstrap replicate of $\mathrm{AVaR}_{0.1}$")
ax.set_ylabel(f"count (out of {n_bootstrap} resamples)")
ax.set_title("The exact max sits in the far tail of the replicate distribution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig_05_bootstrap_replicate_distribution.png", dpi=140)
plt.show()



## 3. The optimizer's curse experiment

Repeatedly: draw a small finite sample from a *known* Gaussian
population, solve the AVaR-efficient LP on that sample, and compare the
naive in-sample estimate, the bootstrap-sup estimate, and the
bootstrap-80%-quantile estimate, against the *true* population AVaR of
the resulting portfolio.


In [ ]:

rng = np.random.default_rng(4)
true_mu = np.array([0.08, 0.05])
true_sigma = np.array([[0.05, 0.01], [0.01, 0.03]])

result = optimizer_curse_experiment(
    true_mu, true_sigma,
    sample_size=50,       # deliberately small: the curse is strongest with scarce data
    level=0.1,
    target_return=0.03,
    n_bootstrap=200,
    n_trials=400,
    rng=rng,
    quantile_confidence=0.8,
)

print(f"Naive in-sample estimate:   mean bias (true-est) = {result['naive_bias']:+.4f}   "
      f"mean |bias| = {result['naive_mean_abs_bias']:.4f}")
print(f"Bootstrap-sup (coherent):   mean bias (true-est) = {result['sup_bias']:+.4f}   "
      f"mean |bias| = {result['sup_mean_abs_bias']:.4f}")
print(f"Bootstrap-80%-quantile:     mean bias (true-est) = {result['quantile_bias']:+.4f}   "
      f"mean |bias| = {result['quantile_mean_abs_bias']:.4f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

labels = ["naive", "bootstrap-sup\n(coherent)", "bootstrap-q80\n(not proven coherent)"]
mean_abs_biases = [result["naive_mean_abs_bias"], result["sup_mean_abs_bias"], result["quantile_mean_abs_bias"]]
colors = ["C0", "C3", "C1"]
axes[0].bar(labels, mean_abs_biases, color=colors)
axes[0].set_ylabel("mean |true AVaR - estimate|")
axes[0].set_title("Mean absolute bias across 400 trials")

mean_biases = [result["naive_bias"], result["sup_bias"], result["quantile_bias"]]
axes[1].bar(labels, mean_biases, color=colors)
axes[1].axhline(0, color="k", linewidth=0.8)
axes[1].set_ylabel("mean signed bias (true - estimate)")
axes[1].set_title("Signed bias: naive underestimates, sup overcorrects")

plt.tight_layout()
plt.savefig("fig_05_optimizer_curse_bias.png", dpi=140)
plt.show()



## Honest summary of the finding

- The **naive** in-sample AVaR of the sample-optimised portfolio
  systematically **understates** its true (population) risk: a positive
  mean bias, the numerical signature of the optimizer's curse.
- The **bootstrap-sup** estimate, the only one covered by
  Proposition 6.1's coherence guarantee, **overcorrects**: its mean bias
  flips sign and its mean *absolute* bias is typically larger than the
  naive estimator's, because taking an exact maximum over many bootstrap
  replicates is dominated by rare, unusually adverse resamples (visible
  directly in the histogram of Section 2 above).
- The **bootstrap-quantile** version at a moderate confidence level
  (0.8), while **not** provably coherent, achieves in this experiment the
  best reduction in mean absolute bias among the variants considered.

This asymmetry between the two robustified variants — one with a clean
theoretical guarantee but weak practical calibration, the other
practically well-calibrated but without the same guarantee — is reported
here as an honest empirical finding of this thesis, not oversold as a
theorem; it is precisely the sort of trade-off a full theoretical
treatment of distributionally-robust coherent risk measures would need to
resolve, and is flagged as a direction for future work in the Discussion
chapter.
